# Lab 09 - Prompt Chaining with LangChain LCEL

**Week 3 - Prompt Engineering and Task-to-Prompt Mapping (LangChain basics)**

You will decompose a support triage task into a multi step pipeline and build it with
LangChain Expression Language (LCEL). The pattern is the point: small, typed, testable
steps compose into a chain the same way you would compose functions.

**By the end you can:**
1. Explain why chaining beats one giant prompt: typed intermediate results give you
   measurable checkpoints and cheaper debugging.
2. Compose LCEL pipelines with the pipe operator: `prompt | model | parser`.
3. Validate model output against a Pydantic schema and drive an automatic repair loop.
4. Run independent steps in parallel with `RunnableParallel` and merge the results.
5. Add resilience with `.with_retry(...)` and a request timeout.

**Scenario.** A support desk turns raw customer emails into executive briefs. The data is
synthetic and the product (ClipForge) is fictional.

**Running this lab.** Every LLM call goes through `make_model()`, which supports three backends selected with the `LAB09_BACKEND` environment variable (or by editing `BACKEND_MODE` in the setup cell): `lmstudio` for an LM Studio server on port 1234, `ollama` for an Ollama server on port 11434, and `fake` for a deterministic offline model that needs no server. The default, `auto`, tries LM Studio, then Ollama, then the offline fake, so the notebook always runs and self checks even with no server running. LM Studio and Ollama both speak the OpenAI protocol, so the same code path serves both; only the port and model name change. To use a real local model, start the server, load or pull a chat model, set `LAB09_BACKEND`, then re run from the top.

**How checks work.** Each task has a soft `check(...)` cell that prints PASS or FAIL and never
crashes the notebook. Fill each TODO until its checks pass. Fresh notebook opens mostly red.


**Verified stack (build sandbox):** langchain 1.3.13, langchain-core 1.4.9,
langchain-openai 1.3.5, pydantic 2.13.4, openai 2.45.0, python-dotenv 1.2.2. Cohort runtime
target is Python 3.13. Confirm these match the cohort image before class.

## Part 0 - Setup and scaffolding
Run these cells as given. They provide the data, schemas, model factory, parsers, prompts, and the check helper.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from __future__ import annotations
import os, json, socket
from typing import Optional, Literal, List

from pydantic import BaseModel, Field, ValidationError
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

import langchain_core, langchain_openai
print("langchain-core", langchain_core.__version__, "| langchain-openai", langchain_openai.__version__)

In [ ]:
# Soft check helper. It never raises, so the notebook runs top to bottom even while
# TODOs are unfinished. Track a running PASS / FAIL tally.
_RESULTS = {"pass": 0, "fail": 0}

def check(label, cond_fn):
    """Run cond_fn() and record a PASS or FAIL. Any exception counts as FAIL."""
    try:
        ok = bool(cond_fn())
    except Exception as e:
        print(f"[FAIL] {label}  ({type(e).__name__}: {e})")
        _RESULTS["fail"] += 1
        return
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
    _RESULTS["pass" if ok else "fail"] += 1

def score():
    total = _RESULTS["pass"] + _RESULTS["fail"]
    print(f"\nSCORE: {_RESULTS['pass']} / {total} checks passing")

In [ ]:
# Synthetic, clearly fictional support desk data. ClipForge is an invented product.
SUPPORT_EMAILS = [
    {"id": "E01", "email": "Subject: Crash on export\nBody: Hi team, ClipForge v2.3 crashes when I export a 4K timeline. GPU: RTX 2060. Deadline is Friday, can you help? - Alex (E01)"},
    {"id": "E02", "email": "Subject: Feature idea\nBody: Would love batch rename for 120 clips. Also keyboard shortcuts for labels would be amazing. Thanks! - Morgan (E02)"},
    {"id": "E03", "email": "Subject: Billing question\nBody: I was charged twice for May, invoice 77831. Please advise. - Priya (E03)"},
]
print(f"{len(SUPPORT_EMAILS)} synthetic emails loaded")

In [ ]:
class Metadata(BaseModel):
    product_version: Optional[str] = None
    hardware: Optional[str] = None
    invoice: Optional[str] = None

class TicketExtract(BaseModel):
    id: str
    issue_type: Literal["bug_report", "feature_request", "billing", "other"]
    title: str
    details: str
    priority: Literal["low", "medium", "high"]
    metadata: Optional[Metadata] = None

class ExecBrief(BaseModel):
    id: str
    headline: str = Field(max_length=110)
    summary_md: str
    next_actions: List[str]

class RiskAssessment(BaseModel):
    risk_score: int = Field(ge=1, le=5)
    why: str

print("Schemas defined. JSON Schema is derivable, e.g. TicketExtract.model_json_schema()")

In [ ]:
# A deterministic, content aware stand in for a local chat model. It lets this lab
# run and self check offline. make_model() uses a real local server (LM Studio or
# Ollama) instead whenever one is selected and reachable.
_CANNED = {
    "E01": {"id": "E01", "issue_type": "bug_report", "title": "Crash on 4K export",
            "details": "ClipForge v2.3 crashes exporting a 4K timeline on an RTX 2060.",
            "priority": "high", "metadata": {"product_version": "v2.3", "hardware": "RTX 2060"}},
    "E02": {"id": "E02", "issue_type": "feature_request", "title": "Batch rename and label shortcuts",
            "details": "Requests batch rename for 120 clips and keyboard shortcuts for labels.",
            "priority": "low", "metadata": {}},
    "E03": {"id": "E03", "issue_type": "billing", "title": "Double charge for May",
            "details": "Customer charged twice for May, invoice 77831.",
            "priority": "medium", "metadata": {"invoice": "77831"}},
}
_SUMM = {
    "E01": {"id": "E01", "headline": "4K export crash blocks a user ahead of a Friday deadline",
            "summary_md": "- ClipForge v2.3 crashes on 4K export\n- GPU is an RTX 2060\n- Deadline is Friday",
            "next_actions": ["Reproduce with a 4K timeline", "Inspect the GPU export path"]},
    "E02": {"id": "E02", "headline": "Feature request: batch rename and label keyboard shortcuts",
            "summary_md": "- Batch rename for 120 clips\n- Keyboard shortcuts for labels",
            "next_actions": ["Add to backlog", "Estimate effort"]},
    "E03": {"id": "E03", "headline": "Duplicate May charge needs a refund review",
            "summary_md": "- Customer double charged for May\n- Invoice 77831",
            "next_actions": ["Verify invoice 77831", "Issue a refund if confirmed"]},
}
_TITLES = {"E01": "Crash exporting 4K timeline on RTX 2060",
           "E02": "Batch rename and label shortcuts",
           "E03": "Duplicate charge for May invoice 77831"}
_RISK = {"E01": 4, "E02": 2, "E03": 3}

class FakeSupportModel(BaseChatModel):
    @property
    def _llm_type(self) -> str:
        return "fake-support-model"

    def _generate(self, messages: List[BaseMessage], stop=None, run_manager=None, **kwargs) -> ChatResult:
        text = "\n".join(m.content for m in messages if isinstance(m.content, str))
        eid = next((k for k in ("E01", "E02", "E03") if k in text), "E01")
        if "risk_score" in text:
            out = json.dumps({"risk_score": _RISK[eid], "why": "Based on stated priority and urgency."})
        elif "executive brief" in text.lower() or "headline" in text:
            out = json.dumps(_SUMM[eid])
        elif "ticket title" in text.lower():
            out = _TITLES[eid]
        elif "repair" in text.lower():
            out = json.dumps(_CANNED[eid])
        else:
            out = json.dumps(_CANNED[eid])
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=out))])

# ---- Backend selection ------------------------------------------------------
# Pick a model backend with the LAB09_BACKEND environment variable, or by editing
# BACKEND_MODE below. Four values are supported:
#   "auto"      probe LM Studio, then Ollama, else fall back to the offline fake (default)
#   "lmstudio"  force the LM Studio server (default http://localhost:1234/v1)
#   "ollama"    force the Ollama server (default http://localhost:11434/v1)
#   "fake"      force the deterministic offline model, no server, always reproducible
BACKEND_MODE = os.getenv("LAB09_BACKEND", "ollama").lower()

# LM Studio and Ollama both speak the OpenAI protocol, so one ChatOpenAI code path
# serves both. Only the port, the default model name, and the placeholder key differ.
# Each field can be overridden with the matching environment variable.
_SERVERS = {
    "lmstudio": {"port": 1234,  "base_url": "http://localhost:1234/v1",
                 "base_env": "LMSTUDIO_BASE_URL",
                 "model_env": "LMSTUDIO_MODEL", "default_model": "local-model",
                 "key_env": "LMSTUDIO_API_KEY", "default_key": "lm-studio"},
    "ollama":   {"port": 11434, "base_url": "http://localhost:11434/v1",
                 "base_env": "OLLAMA_BASE_URL",
                 "model_env": "OLLAMA_MODEL", "default_model": "gemma4",
                 "key_env": "OLLAMA_API_KEY", "default_key": "ollama"},
}

def _server_up(host="localhost", port=1234, timeout=0.25) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def _openai_backend(name: str, temperature: float, timeout: int):
    from langchain_openai import ChatOpenAI
    cfg = _SERVERS[name]
    model = ChatOpenAI(
        model=os.getenv(cfg["model_env"], cfg["default_model"]),
        temperature=temperature, timeout=timeout, max_retries=0,
        base_url=os.getenv(cfg["base_env"], cfg["base_url"]),
        api_key=os.getenv(cfg["key_env"], cfg["default_key"]),
    )
    return model, name

def make_model(temperature: float = 0.0, timeout: int = 30, mode: Optional[str] = None):
    """Return (model, backend_name) for the selected backend.

    mode defaults to BACKEND_MODE. "auto" tries the local servers in order and
    falls back to the offline fake so the notebook always runs. An explicit
    "lmstudio" or "ollama" that is not reachable raises RuntimeError with a fix, so a
    misconfigured live demo fails loudly instead of silently using the fake.
    """
    mode = (mode or BACKEND_MODE).lower()
    if mode == "fake":
        return FakeSupportModel(), "fake"
    if mode in _SERVERS:
        cfg = _SERVERS[mode]
        if _server_up(port=cfg["port"]):
            return _openai_backend(mode, temperature, timeout)
        raise RuntimeError(
            f"Backend {mode!r} selected but nothing answered on port {cfg['port']}. "
            "Start that server, or set LAB09_BACKEND=fake for the offline model."
        )
    if mode == "auto":
        for name in ("lmstudio", "ollama"):
            if _server_up(port=_SERVERS[name]["port"]):
                return _openai_backend(name, temperature, timeout)
        return FakeSupportModel(), "fake"
    raise ValueError(
        f"Unknown LAB09_BACKEND {mode!r}. Use one of: auto, lmstudio, ollama, fake."
    )

model, BACKEND = make_model()
print(f"backend mode: {BACKEND_MODE}  ->  active backend: {BACKEND}")


In [ ]:
# One parser per output schema. JsonOutputParser(pydantic_object=...) does two jobs:
# it emits schema aware format instructions for the prompt, and it parses and strips
# any code fences the model returns.
ticket_parser = JsonOutputParser(pydantic_object=TicketExtract)
brief_parser  = JsonOutputParser(pydantic_object=ExecBrief)
risk_parser   = JsonOutputParser(pydantic_object=RiskAssessment)
print("format_instructions preview:\n", ticket_parser.get_format_instructions()[:180], "...")

In [ ]:
# Schema is injected as the {format_instructions} value, never pasted as literal JSON.
# Literal braces in a from_template string are parsed as variables and would crash.
EXTRACT_PROMPT = ChatPromptTemplate.from_template(
    "You are an information extraction assistant for a customer support desk.\n"
    "Read the support EMAIL and return one JSON object describing the ticket.\n\n"
    "{format_instructions}\n\n"
    "Rules:\n- Use only facts present in the email. Do not invent values.\n"
    "- If a metadata field is unknown, omit it.\n\nEMAIL:\n{email}\n"
).partial(format_instructions=ticket_parser.get_format_instructions())

REPAIR_PROMPT = ChatPromptTemplate.from_template(
    "You are a JSON repair assistant. The ORIGINAL JSON failed validation.\n"
    "Return a corrected JSON object that satisfies the schema. Keep values that are\n"
    "already valid. For a missing required field that cannot be inferred, use a safe\n"
    "default (issue_type other, priority low).\n\n{format_instructions}\n\n"
    "VALIDATION_ERRORS:\n{errors}\n\nORIGINAL:\n{original}\n"
).partial(format_instructions=ticket_parser.get_format_instructions())

SUMMARIZE_PROMPT = ChatPromptTemplate.from_template(
    "You are an executive brief writer. From the validated TICKET JSON, produce a\n"
    "compact executive brief as JSON with a headline (max 110 chars), a summary_md of\n"
    "2 to 4 short markdown bullet lines, and 2 or 3 next_actions.\n\n"
    "{format_instructions}\n\nTICKET:\n{ticket}\n"
).partial(format_instructions=brief_parser.get_format_instructions())

TITLE_PROMPT = ChatPromptTemplate.from_template(
    "Given the TICKET JSON, propose one concise ticket title of at most 80 characters.\n"
    "Return plain text only, no quotes.\n\nTICKET:\n{ticket}\n"
)

RISK_PROMPT = ChatPromptTemplate.from_template(
    "Given the TICKET JSON, assess delivery risk.\n\n{format_instructions}\n\nTICKET:\n{ticket}\n"
).partial(format_instructions=risk_parser.get_format_instructions())
print("5 prompts ready")

## Part 1 - Why chain? Start with one typed step
A single prompt that does everything is hard to test. Begin with just extraction: raw email in, a typed ticket out. This one step is already independently checkable.

In [ ]:
# TODO 1  (LCEL composition)
# Contract: extract_chain.invoke({"email": <str>}) returns a dict shaped like TicketExtract.
# You are given EXTRACT_PROMPT, model, and ticket_parser. Combine them into ONE runnable.
extract_chain = None  # replace None with your composition

In [ ]:
check("extract_chain returns a dict with an id", lambda: isinstance(extract_chain.invoke({"email": SUPPORT_EMAILS[0]["email"]}), dict))
check("extraction validates against TicketExtract",
      lambda: TicketExtract.model_validate(extract_chain.invoke({"email": SUPPORT_EMAILS[0]["email"]})) is not None)

## Part 2 - Validate the output
Models return text, not guarantees. Validation is your unit test for model behavior. Turn a Pydantic failure into a readable error list you can act on.

In [ ]:
# TODO 2  (validation)
# Contract: validate_ticket(d) returns [] when d satisfies TicketExtract, otherwise a
# list of readable error strings. Do not print, do not raise.
def validate_ticket(d: dict) -> List[str]:
    raise NotImplementedError("TODO 2")

In [ ]:
check("valid ticket yields no errors", lambda: validate_ticket(_CANNED["E01"]) == [])
check("bad enum and missing field are caught",
      lambda: len(validate_ticket({"id": "E03", "issue_type": "WRONG", "title": "x", "details": "y"})) >= 2)

## Part 3 - Repair loop
When extraction is malformed, do not fail. Feed the errors back to the model and ask for a corrected object, then revalidate. This is self correction with a schema as the referee.

In [ ]:
# TODO 3  (repair loop)
# Build repair_chain from REPAIR_PROMPT, model, ticket_parser.
# Then validate_or_repair(ticket): return it unchanged when valid; otherwise invoke the
# repair chain (it expects "errors" and "original"), revalidate, and fall back to the
# original if the repaired object still does not validate.
repair_chain = None
def validate_or_repair(ticket: dict) -> dict:
    raise NotImplementedError("TODO 3")

In [ ]:
check("valid ticket passes through repair unchanged",
      lambda: validate_or_repair(_CANNED["E02"]) == _CANNED["E02"])
check("broken ticket becomes valid after repair",
      lambda: validate_ticket(validate_or_repair({"id": "E03", "issue_type": "WRONG", "title": "x", "details": "y"})) == [])

## Part 4 - Compose the full pipeline
Chain the steps: extract, validate or repair, then summarize into an executive brief. Note the small reshape step between stages: the summarize prompt wants a `ticket` variable.

In [ ]:
# TODO 4  (compose the full pipeline)
# Build summarize_chain from SUMMARIZE_PROMPT, model, brief_parser.
# Then build pipeline so that pipeline.invoke(<email record>) flows:
#   extract -> validate_or_repair -> (reshape into {"ticket": <json str>}) -> summarize
# The summarize step needs a "ticket" variable, so you must reshape between steps.
summarize_chain = None
pipeline = None

In [ ]:
check("pipeline yields one brief per email", lambda: len(pipeline.batch(SUPPORT_EMAILS)) == len(SUPPORT_EMAILS))
check("every brief validates against ExecBrief",
      lambda: all(ExecBrief.model_validate(b) for b in pipeline.batch(SUPPORT_EMAILS)))

## Part 5 - Parallel enrichment
A suggested title and a risk score are independent, so run them side by side with `RunnableParallel`, then merge. This is where LCEL pays off versus hand written control flow.

In [ ]:
# TODO 5  (parallel enrichment)
# Build title_chain (TITLE_PROMPT | model | StrOutputParser) and
# risk_chain (RISK_PROMPT | model | risk_parser).
# Combine them into enrich_parallel so one invoke returns {"title": ..., "risk": ...}.
# Then enrich(ticket) runs both and returns the ticket plus "suggested_title" and "risk".
title_chain = None
risk_chain = None
enrich_parallel = None
def enrich(ticket: dict) -> dict:
    raise NotImplementedError("TODO 5")

In [ ]:
check("enrich adds a non empty suggested_title",
      lambda: bool(enrich(_CANNED["E01"])["suggested_title"]))
check("enrich adds a risk_score in 1..5",
      lambda: 1 <= enrich(_CANNED["E01"])["risk"]["risk_score"] <= 5)

## Part 6 - Resilience: retries and timeouts
Local servers stall. Wrap a chain with `.with_retry(...)` so transient failures are retried, and rely on the per request timeout set in `make_model`.

In [ ]:
# TODO 6  (resilience)
# Wrap extract_chain so it retries up to 3 times on failure. Assign to robust_extract.
# (Look for the runnable method that adds retry behavior. The timeout is already set in
# make_model, so you do not need to touch that here.)
robust_extract = None

In [ ]:
check("robust_extract still returns a valid ticket",
      lambda: TicketExtract.model_validate(robust_extract.invoke({"email": SUPPORT_EMAILS[2]["email"]})) is not None)
check("robust_extract is a retrying runnable", lambda: type(robust_extract).__name__ == "RunnableRetry")

## Stretch A - Batch the whole set
Process every email and confirm all briefs and risk scores validate. `pipeline.batch(...)` runs the records together.

In [ ]:
# STRETCH A  (optional)
# Produce all_enriched (enriched ticket per email) and all_briefs (exec brief per email)
# for the full SUPPORT_EMAILS set, then let the checks below confirm they all validate.
all_enriched = None
all_briefs = None

In [ ]:
check("all briefs validate", lambda: all(ExecBrief.model_validate(b) for b in all_briefs))
check("all risk scores in range", lambda: all(1 <= e["risk"]["risk_score"] <= 5 for e in all_enriched))

## Stretch B - Add a deterministic tool
Not everything needs a model. Put a rule based classifier before the model and let it override priority. Hybrid pipelines keep deterministic work deterministic.

In [ ]:
# STRETCH B  (optional)
# Write priority_policy(email_text) -> one of "low"/"medium"/"high" using simple keyword
# rules, then extract_with_policy(record) that extracts, repairs, and overrides priority
# with the deterministic policy. This shows a hybrid deterministic plus model pipeline.
def priority_policy(email_text: str) -> str:
    raise NotImplementedError("STRETCH B")
def extract_with_policy(record: dict) -> dict:
    raise NotImplementedError("STRETCH B")

In [ ]:
check("policy flags the crash email high", lambda: priority_policy(SUPPORT_EMAILS[0]["email"]) == "high")
check("policy override lands on the ticket",
      lambda: extract_with_policy(SUPPORT_EMAILS[2]["email"] and SUPPORT_EMAILS[2])["priority"] in {"low","medium","high"})

## Wrap up
You built a typed, testable, resilient chain. The same shape scales: swap the fake model for
LM Studio, add steps, or replace a step with a fine tuned model, and the checks still guard you.

**Self assessment (target 12 of 18):** clear why chain rationale; extraction validates; repair
triggers only when needed; full pipeline emits valid briefs; parallel enrichment merges title
and risk; retry and timeout in place. Run the final score cell below.

In [ ]:
score()